# MMDP-OD-RTDP Master Report

This notebook serves as the master empirical report for comparing **Baseline Real-Time Dynamic Programming (RTDP)** against **Operator Decomposition (OD) RTDP** in stochastic Multi-Agent Pathfinding environments (MMDP).

We will progress through increasingly difficult maps. For each map, we will benchmark both planners, and then launch an interactive visualization to watch the OD planner execute alongside its branching tree.

### 1. Framework Setup
First, we initialize the highly optimized batch framework and define our benchmarking function.

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

REPO_URL = "https://github.com/t-rays/MMDP-OD-RTDP-PROJECT.git"
REPO_NAME = "MMDP-OD-RTDP-PROJECT"

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    sys.path.append('src')
else:
    src_path = str(Path('.').resolve() / 'src')
    if src_path not in sys.path:
        sys.path.append(src_path)
        
from grid_mmdp import GridMMDP, MMDPConfig
from map_creator import create_map_instance
from heuristic import ShortestPathHeuristic
from baseline_rtdp import BaselineRTDP, RTDPConfig
from od_rtdp import OperatorDecompositionRTDP
from evaluation import evaluate_policy, EvaluationConfig
from resource_monitor import ResourceMonitor
from visualizer import TrajectoryVisualizer

# Global array to aggregate results across all notebook cells
GLOBAL_RESULTS = []

def run_comparison(map_path: str, n_agents: int, time_limit: float = 5.0, episodes: int = 50):
    map_instance = create_map_instance(map_path, scenario_number=1, task_offset=0, n_agents=n_agents)
    mdp = GridMMDP(map_instance, MMDPConfig(slip_to_stay_probability=0.20))
    heuristic = ShortestPathHeuristic(mdp)
    
    print(f"\n{'='*60}\nRunning Benchmark: {map_instance.grid_map.name} ({n_agents} Agents) | Time Limit: {time_limit}s\n{'='*60}")
    
    config = RTDPConfig(time_limit_seconds=time_limit, seed=42)
    eval_config = EvaluationConfig(episodes=episodes, seed=101)
    
    # Baseline RTDP
    baseline_planner = BaselineRTDP(mdp, heuristic, config)
    print("Running Baseline RTDP...")
    with ResourceMonitor() as monitor:
        baseline_result = baseline_planner.solve()
    baseline_mem = monitor.snapshot().peak_rss_delta_mb
    baseline_eval = evaluate_policy(mdp, baseline_planner, eval_config)
    
    # OD RTDP
    od_planner = OperatorDecompositionRTDP(mdp, heuristic, config)
    print("Running OD RTDP...")
    with ResourceMonitor() as monitor:
        od_result = od_planner.solve()
    od_mem = monitor.snapshot().peak_rss_delta_mb
    od_eval = evaluate_policy(mdp, od_planner, eval_config)
    
    # Print Mini Summary
    print(f"\n[BASELINE] Success: {baseline_eval.summary.success_rate*100:.1f}% | Backups: {baseline_result.bellman_backups:,} | RAM: {baseline_mem:.1f} MB")
    print(f"[OD RTDP]  Success: {od_eval.summary.success_rate*100:.1f}% | Backups: {od_result.bellman_backups:,} | RAM: {od_mem:.1f} MB")
    
    metrics = {
        "map": map_instance.grid_map.name,
        "agents": n_agents,
        "baseline": {
            "trials": baseline_result.trials_completed,
            "backups": baseline_result.bellman_backups,
            "success": baseline_eval.summary.success_rate,
            "steps": baseline_eval.summary.mean_steps_successful_episodes,
            "memory_mb": baseline_mem
        },
        "od": {
            "trials": od_result.trials_completed,
            "backups": od_result.bellman_backups,
            "success": od_eval.summary.success_rate,
            "steps": od_eval.summary.mean_steps_successful_episodes,
            "memory_mb": od_mem
        }
    }
    
    return metrics, mdp, od_planner

print("✅ Framework loaded successfully.")

## Experiment 1: The Curse of Dimensionality (Agent Scaling)
We test an open 8x8 grid with $n=2, 3, 4$ agents. As we add agents, Baseline RTDP's branching factor $|A|^n$ explodes (25 $\to$ 125 $\to$ 625), whereas OD RTDP's branching factor $|A| \times n$ grows linearly (10 $\to$ 15 $\to$ 20).

In [ ]:
for agents in [2, 3, 4]:
    metrics, mdp, planner = run_comparison('maps/empty-8-8', n_agents=agents, time_limit=5.0)
    GLOBAL_RESULTS.append(metrics)

print("\n--- Visualizing OD Execution (4 Agents) ---")
viz_1 = TrajectoryVisualizer(mdp, planner)
viz_1.show_with_tree()

## Experiment 2: Diagnostic Crossing 9x9 (3 Agents)
This map forces the 3 agents to cross through a central intersection, creating artificial conflict bottlenecks.

In [ ]:
metrics, mdp, planner = run_comparison('maps/diagnostic/crossing-9-9', n_agents=3, time_limit=5.0)
GLOBAL_RESULTS.append(metrics)

print("\n--- Visualizing OD Execution ---")
viz_2 = TrajectoryVisualizer(mdp, planner)
viz_2.show_with_tree()

## Experiment 3: Complex Warehouse (3 Agents)
This is a massive warehouse structure with long corridors. We allow 10 seconds of planning time to handle the vast state space.

In [ ]:
metrics, mdp, planner = run_comparison('maps/warehouse-10-20-10-2-1', n_agents=3, time_limit=10.0)
GLOBAL_RESULTS.append(metrics)

print("\n--- Visualizing OD Execution ---")
viz_3 = TrajectoryVisualizer(mdp, planner)
viz_3.show_with_tree()

## Final Aggregation & Analysis
With all map problems completed, we now aggregate the metrics collected in the global array to visualize the massive performance differences between Baseline and Operator Decomposition RTDP.

In [ ]:
import pandas as pd

rows = []
for res in GLOBAL_RESULTS:
    # Baseline Row
    rows.append({
        'Map': res['map'],
        'Agents': res['agents'],
        'Algorithm': 'Baseline RTDP',
        'Success Rate (%)': res['baseline']['success'] * 100,
        'Bellman Backups': res['baseline']['backups'],
        'Peak RAM (MB)': res['baseline']['memory_mb']
    })
    # OD Row
    rows.append({
        'Map': res['map'],
        'Agents': res['agents'],
        'Algorithm': 'OD RTDP',
        'Success Rate (%)': res['od']['success'] * 100,
        'Bellman Backups': res['od']['backups'],
        'Peak RAM (MB)': res['od']['memory_mb']
    })

df = pd.DataFrame(rows)
display(df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Filter empty-8-8 for scaling lineplots
df_scale = df[df['Map'] == 'empty-8-8'].copy()

# 1. Bellman Backups (Log Scale) Lineplot
sns.lineplot(data=df_scale, x='Agents', y='Bellman Backups', hue='Algorithm', 
             marker='o', linewidth=2, palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('Computation Scaling (empty-8-8)')
axes[0].set_xticks([2, 3, 4])

# 2. Memory Scaling Lineplot
sns.lineplot(data=df_scale, x='Agents', y='Peak RAM (MB)', hue='Algorithm', 
             marker='o', linewidth=2, palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title('Memory Scaling (empty-8-8)')
axes[1].set_xticks([2, 3, 4])

# 3. Success Rate Barplot (all maps, 3 agents)
df_3ag = df[df['Agents'] == 3].copy()
sns.barplot(data=df_3ag, x='Map', y='Success Rate (%)', hue='Algorithm', 
            palette=['#e74c3c', '#2ecc71'], ax=axes[2])
axes[2].set_title('Robustness on Different Maps (3 Agents)')
axes[2].set_ylim(0, 105)
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

## Conclusion

### The Curse of Dimensionality
Baseline RTDP struggles because the branching factor of the joint-action space grows exponentially ($|A|^n$). With 3 agents having 5 moves each, the branching factor is 125. This causes extreme memory pressure and slows down state expansion, severely limiting the number of Bellman backups that can be completed.

### The OD Solution
Operator Decomposition mitigates this by breaking simultaneous joint actions into a sequence of individual decisions. The branching factor drops to $|A| \times n = 15$. As demonstrated by the animated trees above, the state-space tree is much narrower, allowing the planner to compute vastly more Bellman backups using a fraction of the RAM. 

As proven in the charts, this structural advantage consistently yields highly robust policies even on complex bottleneck maps like the Warehouse.

## Appendix: Interactive OD vs Joint Simulator Sandbox
Below is a self-contained pedagogical interactive dashboard. It runs its own stochastic simulator using Value Iteration to compute Q-values, and then lets you step through the timeline to see exactly how the Joint Enumeration (Baseline) tree and the Operator Decomposition (OD) tree are constructed. 

**Use the sliders to tweak the slip probability and move agents around!**

In [ ]:
# =====================================================================
# IMPORTS & DEPENDENCIES
# =====================================================================
import re
import time
import threading
import warnings
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import graphviz
import ipywidgets as widgets
from IPython import display
from typing import Dict, List, Tuple, Any, Optional

warnings.filterwarnings('ignore', category=DeprecationWarning)

# =====================================================================
# PROBLEM DECLARATIONS & CONFIGURATION
# =====================================================================
DEFAULT_MAP_PATH: str = "maps/random-32-32-10.map"
MAX_SIMULATION_STEPS: int = 100
MIN_AGENTS: int = 3
MAX_AGENTS: int = 10
DEFAULT_AGENTS: int = 3

# MDP hyper-parameters
DEFAULT_SLIP: float = 0.20        # total probability of slipping (split over 2 perpendiculars)
GAMMA: float = 0.95               # discount factor
VI_TOL: float = 1e-4              # value-iteration convergence tolerance
VI_MAX_ITERS: int = 1000          # value-iteration iteration cap
LIVING_COST: float = -1.0         # per-step reward while not at goal
ARRIVAL_BONUS: float = 10.0       # one-time reward on first arrival at goal
CONFLICT_PENALTY: float = 1e3     # penalty applied to conflicting joint actions

ACTIONS: Dict[int, str] = {0: 'Up', 1: 'Down', 2: 'Left', 3: 'Right', 4: 'Stay'}
MOVES: Dict[int, Tuple[int, int]] = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1), 4: (0, 0)}
# Perpendicular slip directions for each directional action
PERP: Dict[int, Tuple[int, int]] = {0: (2, 3), 1: (2, 3), 2: (0, 1), 3: (0, 1)}

COLORS: Dict[str, str] = {
    'start_no': '#ff7675',
    'start_with': '#74b9ff',
    'chosen': '#00b894',
    'joint_best': '#e17055',   # orange: true joint argmax when it differs from executed OD action
    'rejected': '#ffeaa7',
    'unselected': '#dfe6e9',
    'agent_2': '#0984e3',
    'agent_3': '#b2bec3',
    'text_light': 'white',
    'text_dark': 'black',
}

# =====================================================================
# MATH FORMATTING UTILITY
# =====================================================================
def format_graphviz_math(raw_label: str) -> str:
    """LaTeX-ish sub/superscripts -> Graphviz HTML label with clean line spacing."""
    html = re.sub(r'\^\{([^}]+)\}', r'<SUP>\1</SUP>', raw_label)
    html = re.sub(r'\^([^{<\s])', r'<SUP>\1</SUP>', html)
    html = re.sub(r'_\{([^}]+)\}', r'<SUB>\1</SUB>', html)
    html = re.sub(r'_([^{<\s])', r'<SUB>\1</SUB>', html)
    lines = html.split('\n')
    table_rows = "".join(f"<TR><TD ALIGN='CENTER'>{line}</TD></TR>" for line in lines)
    return f'<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="2">{table_rows}</TABLE>>'

# =====================================================================
# ENVIRONMENT ENGINE (STOCHASTIC "SLIPPERY" MAZE SIMULATOR)
# =====================================================================
class MazeEnvironment:
    """
    Multi-agent slippery gridworld.
      * Stochastic transitions: intended move w.p. (1 - slip); each of the two
        perpendicular moves w.p. slip/2. 'Stay' never slips. Invalid outcomes
        (wall / boundary) resolve to staying in place.
      * step() samples the actual (post-slip) outcome, then resolves vertex,
        swap and cascading collisions to a fixed point.
      * One-time arrival bonuses; -1 team living cost per step until done.
    """

    def __init__(self, file_path: str, num_agents: int,
                 slip: float = DEFAULT_SLIP, seed: Optional[int] = None):
        self.num_agents: int = num_agents
        self.slip: float = float(slip)
        self.rng = np.random.default_rng(seed)
        self.height: int = 0
        self.width: int = 0
        self.grid: np.ndarray = np.array([])
        self.agent_pos: Dict[int, List[int]] = {}
        self.goals: Dict[int, List[int]] = {}
        self.trail: np.ndarray = np.array([])
        self.rewarded: set = set()  # agents that already received their arrival bonus
        self._load_map(file_path)

    # ---------------- map loading ----------------

    def _load_map(self, file_path: str) -> None:
        try:
            with open(file_path, 'r') as file:
                map_lines = [line.strip() for line in file.readlines() if line.strip()]
            map_start = 0
            for idx, line in enumerate(map_lines):
                if 'height' in line:
                    self.height = int(line.split()[1])
                if 'width' in line:
                    self.width = int(line.split()[1])
                if line == 'map' or line.startswith('map'):
                    map_start = idx + 1
                    break
            self.grid = np.zeros((self.height, self.width), dtype=int)
            for r in range(self.height):
                for c in range(min(self.width, len(map_lines[map_start + r]))):
                    if map_lines[map_start + r][c] == '@':
                        self.grid[r, c] = 1
        except (FileNotFoundError, IndexError, ValueError):
            print(f"⚠ Map file {file_path} not found/invalid — generating a synthetic 32x32 map (10% obstacles).")
            self.height, self.width = 32, 32
            gen = np.random.default_rng(42)
            self.grid = (gen.random((self.height, self.width)) < 0.10).astype(int)

    # ---------------- placement validation ----------------

    def _nearest_free(self, r: int, c: int, taken: set) -> List[int]:
        """BFS outward from (r, c) to the nearest free, un-taken cell."""
        from collections import deque
        r = int(np.clip(r, 0, self.height - 1))
        c = int(np.clip(c, 0, self.width - 1))
        q, seen = deque([(r, c)]), {(r, c)}
        while q:
            cr, cc = q.popleft()
            if self.grid[cr, cc] == 0 and (cr, cc) not in taken:
                return [cr, cc]
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                nr, nc = cr + dr, cc + dc
                if 0 <= nr < self.height and 0 <= nc < self.width and (nr, nc) not in seen:
                    seen.add((nr, nc))
                    q.append((nr, nc))
        raise RuntimeError("No free cell found on the map.")

    def reset(self) -> Dict[int, List[int]]:
        self.agent_pos.clear()
        self.goals.clear()
        self.rewarded = set()
        taken_starts, taken_goals = set(), set()
        for i in range(self.num_agents):
            r_off, c_off = i // 5, i % 5
            start = self._nearest_free(2 + r_off, 10 + c_off, taken_starts)
            goal = self._nearest_free(2 + r_off, 25 - c_off, taken_goals)
            taken_starts.add(tuple(start))
            taken_goals.add(tuple(goal))
            self.agent_pos[i], self.goals[i] = start, goal
        self.trail = np.zeros((self.height, self.width))
        return self.agent_pos

    def validate_positions(self) -> None:
        """Re-snap user-edited positions onto free cells (called before launch)."""
        taken_s, taken_g = set(), set()
        for i in range(self.num_agents):
            self.agent_pos[i] = self._nearest_free(*self.agent_pos[i], taken_s)
            self.goals[i] = self._nearest_free(*self.goals[i], taken_g)
            taken_s.add(tuple(self.agent_pos[i]))
            taken_g.add(tuple(self.goals[i]))

    # ---------------- stochastic transition model ----------------

    def transition_outcomes(self, action: int) -> List[Tuple[int, float]]:
        """Return [(executed_action, probability), ...] — the slip distribution."""
        if action == 4 or self.slip <= 0.0:
            return [(action, 1.0)]
        p1, p2 = PERP[action]
        return [(action, 1.0 - self.slip), (p1, self.slip / 2.0), (p2, self.slip / 2.0)]

    def _apply_move(self, r: int, c: int, action: int) -> Tuple[int, int]:
        dr, dc = MOVES[action]
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.height and 0 <= nc < self.width and self.grid[nr, nc] != 1:
            return nr, nc
        return r, c  # bump into wall / boundary → stay

    # ---------------- synchronous stochastic step ----------------

    def step(self, intended_actions: List[int]) -> Tuple[Dict[int, List[int]], float, bool, Dict[str, Any]]:
        """
        1) Sample the executed (post-slip) action per agent.
        2) Compute tentative moves (walls/bounds → stay).
        3) Resolve vertex, swap, and cascading collisions to a fixed point.
        4) One-time arrival bonuses + team living cost.
        """
        n = self.num_agents
        executed: List[int] = []
        for i in range(n):
            a = intended_actions[i] if i < len(intended_actions) else 4
            outcomes = self.transition_outcomes(a)
            acts, probs = zip(*outcomes)
            executed.append(int(self.rng.choice(acts, p=probs)))

        origin = {i: tuple(self.agent_pos[i]) for i in range(n)}
        final = {i: self._apply_move(*origin[i], executed[i]) for i in range(n)}
        reverted: set = set()

        # --- fixed-point collision resolution (vertex + swap + cascades) ---
        changed = True
        while changed:
            changed = False
            # Swap (edge) conflicts: i→pos(j) while j→pos(i)
            for i in range(n):
                for j in range(i + 1, n):
                    if final[i] == origin[j] and final[j] == origin[i] and origin[i] != origin[j]:
                        for k in (i, j):
                            if final[k] != origin[k]:
                                final[k] = origin[k]
                                reverted.add(k)
                                changed = True
            # Vertex conflicts: ≥2 agents on the same target cell
            cell_groups: Dict[Tuple[int, int], List[int]] = {}
            for i in range(n):
                cell_groups.setdefault(final[i], []).append(i)
            for cell, group in cell_groups.items():
                if len(group) > 1:
                    for k in group:
                        if final[k] != origin[k]:
                            final[k] = origin[k]
                            reverted.add(k)
                            changed = True

        self.agent_pos = {i: list(final[i]) for i in range(n)}

        # --- trail marking ---
        for i in range(n):
            r, c = self.agent_pos[i]
            self.trail[r, c] = 2.0 + (i * 0.5)

        # --- reward: -1 living cost + one-time arrival bonuses ---
        reward = LIVING_COST
        done = True
        newly_arrived = []
        for i in range(n):
            if self.agent_pos[i] == self.goals[i]:
                if i not in self.rewarded:
                    reward += ARRIVAL_BONUS
                    self.rewarded.add(i)
                    newly_arrived.append(i)
            else:
                done = False

        info = {
            'executed': executed,
            'slipped': [executed[i] != (intended_actions[i] if i < len(intended_actions) else 4)
                        for i in range(n)],
            'reverted': sorted(reverted),
            'newly_arrived': newly_arrived,
        }
        return self.agent_pos, reward, done, info

# =====================================================================
# PLANNER — VALUE ITERATION (BELLMAN) + JOINT / OD ACTION SELECTION
# =====================================================================
class JointPolicy:
    """
    Per-agent value iteration under the slippery transition model, then:
      * OD selection: agents decide sequentially; agent k maximizes
        Q^k(s,a) subject to hard vertex/swap constraints against the cells
        reserved by agents 1..k-1  →  real a^k ~ π^k(·|s, a^{1:k-1}).
      * Joint enumeration: all 5^m joint actions of the first m = min(3, n)
        agents are scored with joint Q = Σ_k Q^k − conflict penalties, so the
        "125 joint actions evaluated" claim in the WITHOUT-OD panel is true.
    """

    def __init__(self, env: MazeEnvironment, gamma: float = GAMMA):
        self.env = env
        self.gamma = gamma
        self.V: Dict[int, np.ndarray] = {}          # per-agent value function (flattened grid)
        self._next_idx: Dict[int, np.ndarray] = {}  # deterministic successor index per action
        self.vi_iters: Dict[int, int] = {}
        self._build_transition_index()

    # ---------- shared deterministic successor table ----------

    def _build_transition_index(self) -> None:
        H, W = self.env.height, self.env.width
        rows, cols = np.divmod(np.arange(H * W), W)
        for a, (dr, dc) in MOVES.items():
            nr, nc = rows + dr, cols + dc
            valid = (nr >= 0) & (nr < H) & (nc >= 0) & (nc < W)
            nr2 = np.where(valid, nr, rows)
            nc2 = np.where(valid, nc, cols)
            flat = nr2 * W + nc2
            wall = self.env.grid.reshape(-1)[flat] == 1
            flat = np.where(wall, np.arange(H * W), flat)  # bump → stay
            self._next_idx[a] = flat

    def _outcomes(self, action: int) -> List[Tuple[int, float]]:
        return self.env.transition_outcomes(action)

    # ---------- Bellman solve ----------

    def solve(self) -> None:
        """V(s) = max_a Σ_{s'} T(s'|s,a) [ -1 + γ V(s') ], goal absorbing at 0."""
        H, W = self.env.height, self.env.width
        S = H * W
        walls = self.env.grid.reshape(-1) == 1
        for k in range(self.env.num_agents):
            g = self.env.goals[k][0] * W + self.env.goals[k][1]
            V = np.zeros(S)
            for it in range(VI_MAX_ITERS):
                Q = np.full((5, S), -np.inf)
                for a in range(5):
                    q = np.zeros(S)
                    for (ex, p) in self._outcomes(a):
                        q += p * (LIVING_COST + self.gamma * V[self._next_idx[ex]])
                    Q[a] = q
                V_new = Q.max(axis=0)
                V_new[g] = 0.0            # absorbing goal
                V_new[walls] = 0.0        # walls are unreachable; pin for stability
                delta = np.max(np.abs(V_new - V))
                V = V_new
                if delta < VI_TOL:
                    break
            self.V[k] = V
            self.vi_iters[k] = it + 1

    # ---------- Q-values at runtime ----------

    def q_values(self, agent: int, pos: List[int]) -> np.ndarray:
        """Q^k(s, ·) = Σ_{s'} T(s'|s,a) [ -1 + γ V^k(s') ] for the 5 actions."""
        W = self.env.width
        s = pos[0] * W + pos[1]
        q = np.zeros(5)
        for a in range(5):
            for (ex, p) in self._outcomes(a):
                q[a] += p * (LIVING_COST + self.gamma * self.V[agent][self._next_idx[ex][s]])
        return q

    # ---------- conflict test on intended (most-likely) targets ----------

    @staticmethod
    def _conflicts(origins: List[Tuple[int, int]], targets: List[Tuple[int, int]]) -> int:
        """Count vertex + swap conflicts among a set of intended moves."""
        n = len(targets)
        count = 0
        for i in range(n):
            for j in range(i + 1, n):
                if targets[i] == targets[j]:
                    count += 1
                elif targets[i] == origins[j] and targets[j] == origins[i] and origins[i] != origins[j]:
                    count += 1
        return count

    def _intended_target(self, pos: List[int], action: int) -> Tuple[int, int]:
        return self.env._apply_move(pos[0], pos[1], action)

    # ---------- the actual decision procedure ----------

    def decide(self, positions: Dict[int, List[int]]) -> Dict[str, Any]:
        n = self.env.num_agents

        # ---- (A) OD: sequential, genuinely conditioned selection ----
        od_actions: List[int] = []
        od_q: List[np.ndarray] = []
        reserved_targets: List[Tuple[int, int]] = []
        reserved_origins: List[Tuple[int, int]] = []
        for k in range(n):
            pos = positions[k]
            if pos == self.env.goals[k]:
                od_actions.append(4)                     # done → hold position
                od_q.append(self.q_values(k, pos))
                reserved_targets.append(tuple(pos))
                reserved_origins.append(tuple(pos))
                continue
            q = self.q_values(k, pos)
            best_a, best_score = 4, -np.inf
            for a in range(5):
                tgt = self._intended_target(pos, a)
                conflict = tgt in reserved_targets
                if not conflict:
                    for (ro, rt) in zip(reserved_origins, reserved_targets):
                        if tgt == ro and rt == tuple(pos) and ro != tuple(pos):
                            conflict = True   # swap with an earlier agent's plan
                            break
                score = q[a] - (CONFLICT_PENALTY if conflict else 0.0)
                if score > best_score:
                    best_score, best_a = score, a
            od_actions.append(best_a)
            od_q.append(q)
            reserved_targets.append(self._intended_target(pos, best_a))
            reserved_origins.append(tuple(pos))

        # ---- (B) Joint enumeration over the first m = min(3, n) agents ----
        m = min(3, n)
        origins_all = [tuple(positions[i]) for i in range(n)]
        fixed_targets = [self._intended_target(positions[i], od_actions[i]) for i in range(m, n)]
        fixed_origins = origins_all[m:]

        joint_scores = np.zeros(5 ** m)
        for combo in itertools.product(range(5), repeat=m):
            idx = sum(a * (5 ** (m - 1 - p)) for p, a in enumerate(combo))
            score = sum(od_q[k][combo[k]] for k in range(m))
            tgts = [self._intended_target(positions[k], combo[k]) for k in range(m)]
            score -= CONFLICT_PENALTY * self._conflicts(
                origins_all[:m] + fixed_origins, tgts + fixed_targets)
            joint_scores[idx] = score
        joint_best_idx = int(np.argmax(joint_scores))
        chosen_idx = sum(od_actions[k] * (5 ** (m - 1 - p)) for p, k in enumerate(range(m)))

        return {
            'od_actions': od_actions,
            'od_q': od_q,
            'reserved_targets': reserved_targets,
            'joint_scores': joint_scores,
            'joint_best_idx': joint_best_idx,
            'chosen_idx': chosen_idx,
            'm': m,
        }

# =====================================================================
# DECISION TREE VISUALIZER
# =====================================================================
class TreeVisualizer:
    """Two Graphviz trees per step: true joint enumeration vs. real OD conditioning."""

    def __init__(self, num_agents: int, slip: float):
        self.num_agents = num_agents
        self.slip = slip
        self.no_od = graphviz.Digraph(format='svg')
        self.with_od = graphviz.Digraph(format='svg')
        self.current_root_no = 'START_NO'
        self.current_root_with = 'START_WITH'
        self._initialize_graphs()

    def _initialize_graphs(self) -> None:
        n = self.num_agents
        title_no = (f'WITHOUT OD — Joint MMDP:  A = A^1 × ... × A^{n},  |A| = 5^{n} = {5**n:,}\n'
                    f'Tree shows the first-3-agent slice: all 125 combos are scored each step with '
                    f'JointQ = Σ_k Q^k − conflict penalty  (slip = {self.slip:.0%}, γ = {GAMMA})')
        self.no_od.attr(rankdir='LR', label=format_graphviz_math(title_no),
                        fontname='Helvetica-Bold', fontcolor='#d63031')
        self.no_od.node('START_NO', format_graphviz_math('s_{0} ∈ S'), shape='box',
                        style='filled', fillcolor=COLORS['start_no'], fontcolor=COLORS['text_light'])

        title_with = (f'WITH OD — Sequential Decomposition:  a_{{t}}^k = argmax_a Q^k(s_{{t}}, a)  '
                      f's.t. no vertex/swap conflict with a_{{t}}^{{1:k-1}}\n'
                      f'Q^k from value iteration:  Q^k(s,a) = Σ_{{s\'}} T(s\'|s,a)[−1 + γ V^k(s\')]  '
                      f'(genuine conditioning — later agents see earlier reservations)')
        self.with_od.attr(rankdir='LR', label=format_graphviz_math(title_with),
                          fontname='Helvetica-Bold', fontcolor='#0984e3')
        self.with_od.node('START_WITH', format_graphviz_math('s_{0} ∈ S'), shape='box',
                          style='filled', fillcolor=COLORS['start_with'], fontcolor=COLORS['text_light'])

    def add_step(self, step: int, decision: Dict[str, Any], info: Dict[str, Any]) -> Tuple[str, str]:
        acts = decision['od_actions']
        a1 = acts[0] if len(acts) > 0 else 4
        a2 = acts[1] if len(acts) > 1 else 4
        a3 = acts[2] if len(acts) > 2 else 4
        self._update_no_od(step, a1, a2, a3, decision, info)
        self._update_with_od(step, a1, a2, a3, decision, info)
        try:
            return (self.no_od.pipe(format='svg').decode('utf-8'),
                    self.with_od.pipe(format='svg').decode('utf-8'))
        except Exception as e:
            banner = (f"<div style='padding:20px; color:#d63031; font-weight:bold; "
                      f"border:2px dashed #d63031; border-radius:8px;'>"
                      f"⚠ Tree rendering failed: {type(e).__name__}: {e}<br>"
                      f"Most likely the Graphviz system binary ('dot') is not installed "
                      f"or not on PATH. See the install hint printed above the map.</div>")
            return banner, banner

    def _update_no_od(self, step: int, a1: int, a2: int, a3: int,
                      decision: Dict[str, Any], info: Dict[str, Any]) -> None:
        chosen_idx = decision['chosen_idx']
        best_idx = decision['joint_best_idx']
        scores = decision['joint_scores']
        next_root = f'L_{step}_{chosen_idx}'

        for i in range(5):
            for j in range(5):
                for k in range(5):
                    loop_idx = (i * 25) + (j * 5) + k
                    is_chosen = (loop_idx == chosen_idx)
                    is_best = (loop_idx == best_idx) and (best_idx != chosen_idx)
                    node_name = f'L_{step}_{loop_idx}'
                    tip = f"({ACTIONS[i]},{ACTIONS[j]},{ACTIONS[k]})  JointQ={scores[loop_idx]:.2f}"

                    if is_chosen:
                        lbl = (f"s_{{{step}}} ∈ S\n"
                               f"a_{{{step-1}}} = ({ACTIONS[a1]}, {ACTIONS[a2]}, {ACTIONS[a3]})\n"
                               f"JointQ = {scores[chosen_idx]:.2f}   (idx {chosen_idx} / 125 evaluated)")
                        if best_idx == chosen_idx:
                            lbl += "\n= true joint argmax ✓"
                        else:
                            lbl += f"\njoint argmax differs (idx {best_idx}, orange)"
                        slips = [f"A{x+1}:{ACTIONS[acts_i]}→{ACTIONS[info['executed'][x]]}"
                                 for x, acts_i in enumerate(decision['od_actions'][:3])
                                 if x < len(info['slipped']) and info['slipped'][x]]
                        if slips:
                            lbl += "\nslipped: " + ", ".join(slips)
                        if self.num_agents > 3:
                            lbl += f"\n(+{self.num_agents-3} agents decided but not drawn)"
                        self.no_od.node(node_name, format_graphviz_math(lbl), shape='box',
                                        style='filled', fillcolor=COLORS['chosen'],
                                        fontsize='9', tooltip=tip)
                    elif is_best:
                        lbl = (f"joint argmax\n({ACTIONS[i]},{ACTIONS[j]},{ACTIONS[k]})\n"
                               f"JointQ = {scores[loop_idx]:.2f}")
                        self.no_od.node(node_name, format_graphviz_math(lbl), shape='box',
                                        style='filled', fillcolor=COLORS['joint_best'],
                                        fontcolor=COLORS['text_light'], fontsize='8', tooltip=tip)
                    else:
                        placeholder = f"{ACTIONS[i][0]}{ACTIONS[j][0]}{ACTIONS[k][0]}"
                        self.no_od.node(node_name, placeholder, shape='circle', style='filled',
                                        fillcolor=COLORS['rejected'], fontsize='5',
                                        width='0.25', tooltip=tip)

                    color = COLORS['chosen'] if is_chosen else (
                        COLORS['joint_best'] if is_best else COLORS['rejected'])
                    width = '3.5' if (is_chosen or is_best) else '1.0'
                    self.no_od.edge(self.current_root_no, node_name, color=color,
                                    penwidth=width, arrowhead='none')

        self.current_root_no = next_root

    def _update_with_od(self, step: int, a1: int, a2: int, a3: int,
                        decision: Dict[str, Any], info: Dict[str, Any]) -> None:
        od_q = decision['od_q']
        reserved = decision['reserved_targets']

        # Agent 1 micro-decision (unconstrained argmax of Q^1)
        for i in range(5):
            is_a1 = (i == a1)
            c1 = COLORS['chosen'] if is_a1 else COLORS['unselected']
            w1 = '3.5' if is_a1 else '1.0'
            node_a1 = f'OD_{step}_a1_{i}'
            tip1 = f"Q^1({ACTIONS[i]}) = {od_q[0][i]:.2f}" if len(od_q) > 0 else ACTIONS[i]

            if is_a1:
                lbl = (f"a^1_{{{step-1}}} = {ACTIONS[a1]} = argmax_a Q^1(s_{{{step-1}}}, a)\n"
                       f"Q^1 = {od_q[0][a1]:.2f}   reserves cell {reserved[0]}")
                self.with_od.node(node_a1, format_graphviz_math(lbl), shape='box',
                                  style='filled', fillcolor=c1, fontsize='8', tooltip=tip1)
            else:
                self.with_od.node(node_a1, ACTIONS[i][0], shape='circle', style='filled',
                                  fillcolor=c1, fontsize='6', width='0.2', tooltip=tip1)
            self.with_od.edge(self.current_root_with, node_a1, color=c1,
                              penwidth=w1, arrowhead='none')

            if not is_a1:
                continue

            # Agent 2 micro-decision (conditioned on agent 1's reservation)
            for j in range(5):
                is_a2 = (j == a2)
                c2 = COLORS['chosen'] if is_a2 else COLORS['unselected']
                w2 = '3.5' if is_a2 else '1.0'
                node_a2 = f'OD_{step}_a2_{j}'
                tip2 = f"Q^2({ACTIONS[j]}) = {od_q[1][j]:.2f}" if len(od_q) > 1 else ACTIONS[j]

                if is_a2:
                    lbl = (f"a^2_{{{step-1}}} = {ACTIONS[a2]} ~ π^2(·|s_{{{step-1}}}, a^1)\n"
                           f"Q^2 = {od_q[1][a2]:.2f}, avoids reserved {reserved[0]}\n"
                           f"reserves cell {reserved[1]}")
                    self.with_od.node(node_a2, format_graphviz_math(lbl), shape='diamond',
                                      style='filled', fillcolor=COLORS['agent_2'],
                                      fontcolor=COLORS['text_light'], fontsize='8', tooltip=tip2)
                else:
                    self.with_od.node(node_a2, ACTIONS[j][0], shape='circle', style='filled',
                                      fillcolor=c2, fontsize='6', width='0.2', tooltip=tip2)
                self.with_od.edge(node_a1, node_a2, color=c2, penwidth=w2, arrowhead='none')

                if not is_a2:
                    continue

                # Agent 3 micro-decision (conditioned on agents 1 & 2)
                for k in range(5):
                    is_a3 = (k == a3)
                    c3 = COLORS['chosen'] if is_a3 else COLORS['agent_3']
                    w3 = '3.5' if is_a3 else '1.0'
                    node_a3 = f'OD_{step}_a3_{k}'
                    tip3 = f"Q^3({ACTIONS[k]}) = {od_q[2][k]:.2f}" if len(od_q) > 2 else ACTIONS[k]

                    if is_a3:
                        lbl = (f"a^3_{{{step-1}}} = {ACTIONS[a3]} ~ π^3(·|s_{{{step-1}}}, a^1, a^2)\n"
                               f"Q^3 = {od_q[2][a3]:.2f}, avoids {{{reserved[0]}, {reserved[1]}}}\n"
                               f"s_{{{step}}} = T(s_{{{step-1}}}, a_{{{step-1}}})  (slip = {self.slip:.0%})")
                        self.with_od.node(node_a3, format_graphviz_math(lbl), shape='box',
                                          style='filled', fillcolor=c3, fontsize='8', tooltip=tip3)
                    else:
                        self.with_od.node(node_a3, ACTIONS[k][0], shape='circle', style='filled',
                                          fillcolor=c3, fontsize='6', width='0.2', tooltip=tip3)
                    self.with_od.edge(node_a2, node_a3, color=c3, penwidth=w3, arrowhead='none')

        self.current_root_with = f'OD_{step}_a3_{a3}'

# =====================================================================
# UI APP & DASHBOARD MANAGER
# =====================================================================
class SimulationApp:
    """IPyWidgets front-end: agent config, slip slider, live run, rewind timeline."""

    def __init__(self):
        self.simulation_history: List[Dict[str, Any]] = []
        self.env: Optional[MazeEnvironment] = None
        self.policy: Optional[JointPolicy] = None

        self.agent_slider = widgets.IntSlider(
            value=DEFAULT_AGENTS, min=MIN_AGENTS, max=MAX_AGENTS, step=1,
            description='🤖 Total Agents:', continuous_update=False)
        self.slip_slider = widgets.FloatSlider(
            value=DEFAULT_SLIP, min=0.0, max=0.5, step=0.05,
            description='🧊 Slip prob:', readout_format='.0%', continuous_update=False)
        self.seed_box = widgets.IntText(value=0, description='🎲 Seed:',
                                        layout=widgets.Layout(width='150px'))
        self.init_env_btn = widgets.Button(description='Initialize Grid & Agents',
                                           button_style='info', icon='cogs',
                                           layout=widgets.Layout(width='220px'))
        self.launch_btn = widgets.Button(description='▶ Solve MDP & Launch',
                                         button_style='success', icon='rocket',
                                         layout=widgets.Layout(width='220px'), disabled=True)

        self.agent_selector = widgets.Dropdown(description='Configure:', disabled=True)
        self.start_r = widgets.IntSlider(description='Start Row:', disabled=True)
        self.start_c = widgets.IntSlider(description='Start Col:', disabled=True)
        self.goal_r = widgets.IntSlider(description='Goal Row:', disabled=True)
        self.goal_c = widgets.IntSlider(description='Goal Col:', disabled=True)

        self.config_panel_output = widgets.Output()
        self.app_output = widgets.Output()
        self.ui_output = widgets.Output()

        self.init_env_btn.on_click(self.initialize_environment)
        self.launch_btn.on_click(self.run_simulation)
        for slider in [self.start_r, self.start_c, self.goal_r, self.goal_c]:
            slider.observe(self.on_position_change, names='value')
        self.agent_selector.observe(self.on_agent_selected, names='value')

    def display(self) -> None:
        top_menu = widgets.HBox(
            [self.agent_slider, self.slip_slider, self.seed_box, self.init_env_btn, self.launch_btn],
            layout=widgets.Layout(padding='15px', border='2px solid #dfe6e9',
                                  border_radius='8px', margin='0 0 10px 0'))
        display.display(top_menu, self.config_panel_output, self.app_output)

    # ---------------- config panel ----------------

    def initialize_environment(self, b) -> None:
        self.env = MazeEnvironment(DEFAULT_MAP_PATH, self.agent_slider.value,
                                   slip=self.slip_slider.value,
                                   seed=int(self.seed_box.value))
        self.env.reset()
        self.policy = None  # force re-solve on launch

        max_h, max_w = max(1, self.env.height - 1), max(1, self.env.width - 1)
        for r_slider in [self.start_r, self.goal_r]:
            r_slider.max = max_h
        for c_slider in [self.start_c, self.goal_c]:
            c_slider.max = max_w

        self.agent_selector.options = [(f"Agent {i+1}", i) for i in range(self.env.num_agents)]
        self.agent_selector.disabled = False
        self.launch_btn.disabled = False
        self.on_agent_selected({'new': 0})
        self.render_config_preview()

    def on_agent_selected(self, change):
        if change['new'] is None or self.env is None:
            return
        idx = change['new']
        self.toggle_sliders_listening(False)
        self.start_r.value, self.start_c.value = self.env.agent_pos[idx]
        self.goal_r.value, self.goal_c.value = self.env.goals[idx]
        self.toggle_sliders_listening(True)
        self.render_config_preview()

    def on_position_change(self, change):
        if self.env is None:
            return
        idx = self.agent_selector.value
        self.env.agent_pos[idx] = [self.start_r.value, self.start_c.value]
        self.env.goals[idx] = [self.goal_r.value, self.goal_c.value]
        self.policy = None  # positions changed → V^k must be re-solved
        self.render_config_preview()

    def toggle_sliders_listening(self, enable: bool):
        for slider in [self.start_r, self.start_c, self.goal_r, self.goal_c]:
            slider.disabled = not enable

    def render_config_preview(self):
        with self.config_panel_output:
            display.clear_output(wait=True)
            coords_box = widgets.VBox([
                widgets.HTML("<h4>📍 Arrange Agent Setup Structure</h4>"),
                self.agent_selector,
                widgets.HBox([widgets.VBox([self.start_r, self.start_c]),
                              widgets.VBox([self.goal_r, self.goal_c])])
            ], layout=widgets.Layout(padding='10px', border='1px dashed #b2bec3', width='45%'))

            plt.ioff()
            fig, ax = plt.subplots(figsize=(6, 3.5))
            render_matrix = np.copy(self.env.grid).astype(float)
            for i in range(self.env.num_agents):
                sr, sc = self.env.agent_pos[i]
                gr, gc = self.env.goals[i]
                if 0 <= sr < self.env.height and 0 <= sc < self.env.width:
                    render_matrix[sr, sc] = 3.0
                    ax.text(sc, sr, f"A{i+1}", va='center', ha='center',
                            color='white', fontweight='bold', fontsize=8)
                if 0 <= gr < self.env.height and 0 <= gc < self.env.width:
                    render_matrix[gr, gc] = 7.0
                    ax.text(gc, gr, f"G{i+1}", va='center', ha='center',
                            color='black', fontweight='bold', fontsize=8)
            ax.imshow(render_matrix, cmap='Accent')
            ax.set_title("Live Position Map Preview (walls auto-corrected at launch)")
            ax.set_xticks([]); ax.set_yticks([])

            preview_out = widgets.Output()
            with preview_out:
                display.display(fig)
                plt.close(fig)
            plt.ion()
            display.display(widgets.HBox([coords_box, preview_out],
                                         layout=widgets.Layout(align_items='center')))

    # ---------------- dashboard html (unchanged zoom logic) ----------------

    @staticmethod
    def build_dashboard_html(no_od_svg: str, with_od_svg: str, border_color: str = "#ddd") -> str:
        return f"""
        <div style="width: 100%; position: relative; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
            <div style="position: sticky; top: 0; left: 0; background: rgba(248, 249, 250, 0.95);
                        padding: 8px 12px; border-radius: 8px; border: 1px solid #ccc;
                        display: flex; gap: 10px; width: fit-content; margin-bottom: 15px;
                        z-index: 10000; align-items: center; box-shadow: 0 4px 6px rgba(0,0,0,0.08);">
                <span style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.5px; font-weight: bold; color: #555;">Zoom Controls:</span>
                <button onclick="changeZoom(-10)" style="font-weight:bold; width:28px; height:28px; border-radius:4px; border:1px solid #bbb; background:#fff; cursor:pointer;">&minus;</button>
                <span id="zoom-label" style="font-size:13px; font-weight:bold; color:#222; min-width: 45px; text-align:center;">---</span>
                <button onclick="changeZoom(10)" style="font-weight:bold; width:28px; height:28px; border-radius:4px; border:1px solid #bbb; background:#fff; cursor:pointer;">+</button>
            </div>
            <div style="display: flex; flex-direction: column; gap: 20px; width: 100%;">
                <div class="scroll-container" style="background-color: white; padding: 15px; border: 2px solid #d63031; border-radius: 8px; max-width: 100%; overflow-x: auto;">
                    {no_od_svg}
                </div>
                <div class="scroll-container" style="background-color: white; padding: 15px; border: 2px solid #0984e3; border-radius: 8px; max-width: 100%; overflow-x: auto;">
                    {with_od_svg}
                </div>
            </div>
        </div>

        <script>
            (function() {{
                let cachedZoom = localStorage.getItem('mazeDashboardZoom');
                let activeZoom = cachedZoom ? parseInt(cachedZoom, 10) : 100;
                if (isNaN(activeZoom) || activeZoom < 10) activeZoom = 10;
                applyZoom(activeZoom);

                function applyZoom(targetValue) {{
                    let styleTag = document.getElementById('dynamic-zoom-style');
                    if (!styleTag) {{
                        styleTag = document.createElement('style');
                        styleTag.id = 'dynamic-zoom-style';
                        document.head.appendChild(styleTag);
                    }}
                    styleTag.innerHTML = '.scroll-container svg {{ width: ' + targetValue + '% !important; max-width: none !important; height: auto !important; }}';
                    let label = document.getElementById('zoom-label');
                    if (label) label.innerText = targetValue + '%';
                    localStorage.setItem('mazeDashboardZoom', targetValue);
                }}

                window.changeZoom = function(amount) {{
                    let current = parseInt(localStorage.getItem('mazeDashboardZoom') || 100, 10);
                    applyZoom(Math.max(10, Math.min(450, current + amount)));
                }};
            }})();
        </script>
        """

    # ---------------- simulation ----------------

    def run_simulation(self, b) -> None:
        if self.env is None:
            return
        with self.app_output:
            display.clear_output(wait=True)
            self.simulation_history.clear()

            # Snap any user-edited positions off walls, reset bonuses/trail
            self.env.validate_positions()
            self.env.rewarded = set()
            self.env.trail = np.zeros((self.env.height, self.env.width))
            self.env.slip = self.slip_slider.value
            self.env.rng = np.random.default_rng(int(self.seed_box.value))

            # --- early environment check: the graphviz SYSTEM binary must exist ---
            try:
                graphviz.version()
            except Exception:
                print("❌ Graphviz system binary ('dot') not found — the branching trees "
                      "cannot render.\n"
                      "   The pip package alone is not enough; install the binary:\n"
                      "   • Colab / Debian / Ubuntu:  !apt-get -qq install -y graphviz\n"
                      "   • Conda:                    conda install -c conda-forge python-graphviz\n"
                      "   • Windows:                  install from graphviz.org, add bin/ to PATH, "
                      "restart the kernel.\n"
                      "   The map simulation will still run; trees will show an error banner.")

            num_agents = self.env.num_agents
            print(f"🧮 Solving {num_agents} per-agent MDPs via value iteration "
                  f"(γ = {GAMMA}, slip = {self.env.slip:.0%}) ...")
            self.policy = JointPolicy(self.env, gamma=GAMMA)
            self.policy.solve()
            iters = ", ".join(f"A{k+1}:{v}" for k, v in self.policy.vi_iters.items())
            print(f"✅ Value iteration converged (Bellman sweeps per agent → {iters})")

            visualizer = TreeVisualizer(num_agents, self.env.slip)

            step_counter, done = False, False
            step_counter = 0
            cumulative_reward = 0.0
            fig, ax = plt.subplots(figsize=(12, 6))

            while not done and step_counter < MAX_SIMULATION_STEPS:
                step_counter += 1

                # ---- real MMDP decision: joint scoring + OD-conditioned selection ----
                decision = self.policy.decide(self.env.agent_pos)

                # ---- stochastic synchronous step with full collision resolution ----
                agents, reward, done, info = self.env.step(decision['od_actions'])
                cumulative_reward += reward

                # ---- render map ----
                ax.clear()
                render_matrix = np.copy(self.env.grid).astype(float)
                walked = (self.env.trail > 0) & (render_matrix == 0)
                render_matrix[walked] = self.env.trail[walked]
                for i in range(num_agents):
                    gr, gc = self.env.goals[i]
                    render_matrix[gr, gc] = 5 + (i % 5)
                    ar, ac = self.env.agent_pos[i]
                    render_matrix[ar, ac] = 2 + (i % 5)
                    ax.text(ac, ar, f"A{i+1}", va='center', ha='center', fontweight='bold',
                            color=COLORS['text_light'], fontsize=7)
                ax.imshow(render_matrix, cmap=matplotlib.colormaps['Paired'].resampled(12))
                ax.set_xticks([]); ax.set_yticks([])

                n_slips = sum(info['slipped'])
                slip_txt = f" | Slips: {n_slips}" if n_slips else ""
                rev_txt = f" | Collisions reverted: {len(info['reverted'])}" if info['reverted'] else ""
                ax.set_title(f"Active Map | {num_agents} Agents | Step {step_counter} | "
                             f"R = {reward:+.1f} (Σ {cumulative_reward:+.1f}){slip_txt}{rev_txt}",
                             fontsize=12, fontweight='bold')

                # ---- update Graphviz trees with the REAL decision data ----
                no_svg, with_svg = visualizer.add_step(step_counter, decision, info)

                self.simulation_history.append({
                    'step': step_counter,
                    'matrix': np.copy(render_matrix),
                    'agents': {k: list(v) for k, v in agents.items()},
                    'no_od_svg': no_svg,
                    'with_od_svg': with_svg,
                })

                display.clear_output(wait=True)
                display.display(fig)
                display.display(display.HTML(self.build_dashboard_html(no_svg, with_svg)))
                time.sleep(1.0)

            plt.close(fig)
            display.clear_output(wait=True)
            status = "all agents reached their goals 🏁" if done else "step limit reached ⏱"
            print(f"Simulation complete — {status}. Steps: {step_counter}, "
                  f"cumulative reward: {cumulative_reward:+.1f}. "
                  f"Inspect the joint/OD branching below.")
            self._render_timeline_ui()

    # ---------------- rewind timeline ----------------

    def _render_timeline_ui(self) -> None:
        max_steps = len(self.simulation_history)
        if max_steps == 0:
            return
        step_slider = widgets.IntSlider(min=1, max=max_steps, step=1, value=max_steps,
                                        description='Step:', continuous_update=False,
                                        layout=widgets.Layout(width='300px'))
        btn_first = widgets.Button(description='⏮', layout=widgets.Layout(width='50px'))
        btn_prev = widgets.Button(description='◀', layout=widgets.Layout(width='50px'))
        btn_next = widgets.Button(description='▶', layout=widgets.Layout(width='50px'))
        btn_last = widgets.Button(description='⏭', layout=widgets.Layout(width='50px'))
        btn_play = widgets.Button(description='▶ Play', layout=widgets.Layout(width='80px'))
        btn_play.is_playing = False

        def play_loop():
            while btn_play.is_playing and step_slider.value < max_steps:
                step_slider.value += 1
                time.sleep(1.0)
            btn_play.is_playing = False
            btn_play.description = '▶ Play'

        def toggle_play(b):
            btn_play.is_playing = not btn_play.is_playing
            if btn_play.is_playing:
                btn_play.description = '⏸ Pause'
                if step_slider.value == max_steps:
                    step_slider.value = 1
                threading.Thread(target=play_loop).start()
            else:
                btn_play.description = '▶ Play'

        btn_play.on_click(toggle_play)
        btn_first.on_click(lambda b: setattr(step_slider, 'value', 1))
        btn_prev.on_click(lambda b: setattr(step_slider, 'value', max(1, step_slider.value - 1)))
        btn_next.on_click(lambda b: setattr(step_slider, 'value', min(max_steps, step_slider.value + 1)))
        btn_last.on_click(lambda b: setattr(step_slider, 'value', max_steps))

        control_deck = widgets.HBox([btn_first, btn_prev, btn_play, step_slider, btn_next, btn_last],
                                    layout=widgets.Layout(align_items='center',
                                                          justify_content='center', margin='10px 0px'))
        step_slider.observe(lambda ch: self._rewind_viewer(ch['new']), names='value')
        display.display(widgets.VBox([control_deck, self.ui_output]))
        self._rewind_viewer(max_steps)   # force-draw the final state immediately

    def _rewind_viewer(self, step_val: int) -> None:
        if not self.simulation_history:
            return

        snap = self.simulation_history[step_val - 1]

        with self.ui_output:
            display.clear_output(wait=True)
            plt.ioff()
            fig_play, ax_play = plt.subplots(figsize=(12, 6))
            ax_play.imshow(snap['matrix'], cmap=matplotlib.colormaps['Paired'].resampled(12))

            for i, pos in snap['agents'].items():
                ax_play.text(pos[1], pos[0], f"A{i+1}", va='center', ha='center',
                             fontweight='bold', color=COLORS['text_light'], fontsize=7)

            ax_play.set_xticks([]); ax_play.set_yticks([])
            ax_play.set_title(f"Rewind Dashboard | Map State at Step {step_val}",
                              fontsize=12, fontweight='bold')

            display.display(fig_play)
            plt.close(fig_play)
            plt.ion()

            # --- Added Display Logic ---
            display.display(display.HTML("<b style='color:#d63031'>WITHOUT OD (joint)</b>"))
            display.display(display.SVG(snap['no_od_svg']))

            display.display(display.HTML("<b style='color:#0984e3'>WITH OD (sequential)</b>"))
            display.display(display.SVG(snap['with_od_svg']))

            # If you still need the dashboard HTML, keep this:
            display.display(display.HTML(self.build_dashboard_html(
                snap['no_od_svg'], snap['with_od_svg'], border_color=COLORS['chosen'])))

app = SimulationApp()
app.display()

